In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
import os

In [ ]:
df = pd.read_parquet("data/qa2_labelled_dataset.parquet")

pos = df[df["label"] == True]
neg = df[df["label"] == False]

n_pos = len(pos)          # all positives (231)
n_neg = 500 - n_pos       # remaining negatives

neg_sample = neg.sample(n=n_neg, random_state=42)

sample = pd.concat([pos, neg_sample]) \
           .sample(frac=1, random_state=42) \
           .reset_index(drop=True)

print(f"Total: {len(sample)}  |  Positive: {sample['label'].sum()}  |  Negative: {(~sample['label']).sum()}")

# Save as parquet instead of CSV
sample.to_parquet("dataset/sampled_labelled_dataset.parquet", index=False)
print("Saved: dataset/sampled_labelled_dataset.parquet")

Total: 500  |  Positive: 231  |  Negative: 269
Saved: dataset/sampled_labelled_dataset.parquet


In [ ]:
from score_experiment import run_score_experiment

run_score_experiment(
    parquet_path = "dataset/sampled_labelled_dataset.parquet",
    method_names = ["mean", "poly_deg1", "poly_deg2",
                    "nwkr_gaussian", "nwkr_laplace", "lrt", "capa", "bocpd", "ruptures_kernelcpd", "stumpy"],
    out_dir      = "data/score_experiment_500",
    max_rows     = None,       # already 500 rows, no need to subsample
    max_workers  = 16,
    balance      = False,      # use as-is
    seed         = 42,
)

Loaded 500 rows  (label=True: 231, label=False: 269)
Scoring 500 rows × 10 methods with 16 workers...


In [ ]:
def run_score_threshold_gridsearch(
    results_path: str,
    score_values: list = None,
    iou_thresh: float = 0.75,
    out_dir: str = "data/score_experiment_500",
) -> pd.DataFrame:
    os.makedirs(out_dir, exist_ok=True)

    if score_values is None:
        score_values = [round(v, 2) for v in np.arange(0.30, 1.01, 0.05)]

    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} rows, {df['method'].nunique()} methods, "
          f"{df['label'].sum()} positive, {(~df['label']).sum()} negative")

    pos_df = df[df["label"] == True]
    neg_df = df[df["label"] == False]

    summary_rows = []
    for thresh in score_values:
        for method, pos_sub in pos_df.groupby("method"):
            neg_sub = neg_df[neg_df["method"] == method]

            tp = int(((pos_sub["score"] >= thresh) &
                       (pos_sub["iou_val"] >= iou_thresh)).sum())
            fn = int(len(pos_sub) - tp)
            fp = int((neg_sub["score"] >= thresh).sum())
            tn = int(len(neg_sub) - fp)

            total     = tp + fp + tn + fn
            accuracy  = (tp + tn) / total        if total > 0   else 0.0
            precision = tp / (tp + fp)           if tp + fp > 0 else 0.0
            recall    = tp / (tp + fn)           if tp + fn > 0 else 0.0
            f1        = (2 * precision * recall / (precision + recall)
                         if precision + recall > 0 else 0.0)

            summary_rows.append({
                "score_thresh":   thresh,
                "iou_thresh":     iou_thresh,
                "method":         method,
                "tp": tp, "fp": fp, "tn": tn, "fn": fn,
                "accuracy":       round(accuracy,  4),
                "precision":      round(precision, 4),
                "recall":         round(recall,    4),
                "f1":             round(f1,        4),
                "mean_score_pos": round(float(pos_sub["score"].mean()), 4),
                "mean_score_neg": round(float(neg_sub["score"].mean()), 4),
            })

    results = pd.DataFrame(summary_rows)

    for metric in ["f1", "precision", "recall", "accuracy"]:
        pivot = results.pivot(index="score_thresh", columns="method", values=metric)
        print(f"\n=== {metric.upper()} ===")
        print(pivot.round(3).to_string())

    out_path = os.path.join(out_dir, f"score_threshold_gridsearch_iou{iou_thresh:.2f}.csv")
    results.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")
    return results


# # --- Run on your pre-computed results ---
# results = run_score_threshold_gridsearch(
#     results_path = "data/score_experiment/score_experiment_results.csv",
#     score_values = [round(v, 2) for v in np.arange(0.30, 1.01, 0.05)],
#     iou_thresh   = 0.75,
#     out_dir      = "data/score_experiment",
# )

# Run for multiple IoU thresholds at once
for iou_t in [0.50, 0.75, 0.90]:
    run_score_threshold_gridsearch(
        results_path = "data/score_experiment_500/score_experiment_results.csv",
        score_values = [round(v, 2) for v in np.arange(0.30, 1.01, 0.05)],
        iou_thresh   = iou_t,
        out_dir      = "data/score_experiment_500",
    )



Loaded 8000 rows, 8 methods, 1848 positive, 6152 negative

=== F1 ===
method         capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2  stumpy
score_thresh                                                                                
0.30          0.210  0.604  0.536          0.972         0.967      0.571      0.651   0.496
0.35          0.176  0.555  0.517          0.976         0.974      0.576      0.623   0.487
0.40          0.157  0.527  0.511          0.980         0.955      0.579      0.638   0.442
0.45          0.136  0.479  0.471          0.890         0.913      0.557      0.592   0.405
0.50          0.090  0.436  0.427          0.819         0.834      0.530      0.554   0.381
0.55          0.067  0.406  0.403          0.736         0.766      0.517      0.548   0.334
0.60          0.050  0.372  0.369          0.631         0.693      0.464      0.544   0.254
0.65          0.034  0.325  0.330          0.568         0.594      0.419      0.505   0.208


In [ ]:
def run_iou_gridsearch_from_csv(
    results_path: str,
    iou_values: list = None,
    score_thresh: float = 0.5,
    out_dir: str = "data/score_experiment_500",
) -> pd.DataFrame:
    os.makedirs(out_dir, exist_ok=True)

    if iou_values is None:
        iou_values = [round(v, 2) for v in np.arange(0.50, 1.01, 0.05)]

    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} rows, {df['method'].nunique()} methods, "
          f"{df['label'].sum()} positive, {(~df['label']).sum()} negative")

    summary_rows = []
    for thresh in iou_values:
        for method, sub in df.groupby("method"):
            pos_rows = sub[sub["label"] == True]
            neg_rows = sub[sub["label"] == False]

            tp = int(((pos_rows["score"] >= score_thresh) &
                       (pos_rows["iou_val"] >= thresh)).sum())
            fn = int(len(pos_rows) - tp)
            fp = int((neg_rows["score"] >= score_thresh).sum())
            tn = int(len(neg_rows) - fp)


            total     = tp + fp + tn + fn
            accuracy  = (tp + tn) / total        if total > 0   else 0.0
            precision = tp / (tp + fp)           if tp + fp > 0 else 0.0
            recall    = tp / (tp + fn)           if tp + fn > 0 else 0.0
            f1        = (2 * precision * recall / (precision + recall)
                         if precision + recall > 0 else 0.0)

            summary_rows.append({
                "iou_thresh":     thresh,
                "score_thresh":   score_thresh,
                "method":         method,
                "tp": tp, "fp": fp, "tn": tn, "fn": fn,
                "accuracy":       round(accuracy,  4),
                "precision":      round(precision, 4),
                "recall":         round(recall,    4),
                "f1":             round(f1,        4),
                "mean_score_pos": round(float(pos_rows["score"].mean()), 4),
                "mean_score_neg": round(float(neg_rows["score"].mean()), 4),
                "median_iou_pos": round(float(pos_rows["iou_val"].median()), 4),
                "mean_time_ms":     round(float(sub["time_ms"].mean()),       3),
                "median_time_ms":   round(float(sub["time_ms"].median()),     3),
            })

    results = pd.DataFrame(summary_rows)

    for metric in ["f1", "precision", "recall", "accuracy"]:
        pivot = results.pivot(index="iou_thresh", columns="method", values=metric)
        print(f"\n=== {metric.upper()} ===")
        print(pivot.round(3).to_string())

    out_path = os.path.join(out_dir, f"iou_gridsearch_score{score_thresh:.2f}.csv")
    results.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")
    return results


# --- Run ---
run_iou_gridsearch_from_csv(
    results_path = "data/score_experiment_500/score_experiment_results.csv",
    iou_values   = [round(v, 2) for v in np.arange(0.50, 1.01, 0.05)],
    score_thresh = 0.3,
    out_dir      = "data/score_experiment_500",
)

Loaded 8000 rows, 8 methods, 1848 positive, 6152 negative

=== F1 ===
method       capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2  stumpy
iou_thresh                                                                                
0.50        0.210  0.604  0.536          0.972         0.967      0.571      0.651   0.496
0.55        0.197  0.596  0.528          0.972         0.967      0.561      0.642   0.492
0.60        0.197  0.596  0.528          0.972         0.963      0.558      0.639   0.484
0.65        0.150  0.584  0.517          0.972         0.953      0.532      0.607   0.454
0.70        0.122  0.580  0.517          0.972         0.939      0.525      0.597   0.450
0.75        0.107  0.576  0.513          0.969         0.930      0.518      0.586   0.428
0.80        0.085  0.542  0.493          0.969         0.910      0.476      0.523   0.359
0.85        0.085  0.520  0.486          0.969         0.882      0.458      0.488   0.256
0.90        0.055  0

,iou_thresh,score_thresh,method,tp,fp,tn,fn,accuracy,precision,recall,f1,mean_score_pos,mean_score_neg,median_iou_pos
0,0.5,0.3,capa,29,16,753,202,0.782,0.6444,0.1255,0.2101,0.1506,0.0274,0.1695
1,0.5,0.3,lrt,106,14,755,125,0.861,0.8833,0.4589,0.6040,0.3452,0.0656,0.8000
2,0.5,0.3,mean,105,56,713,126,0.818,0.6522,0.4545,0.5357,0.4036,0.1496,0.8333
3,0.5,0.3,nwkr_gaussian,223,5,764,8,0.987,0.9781,0.9654,0.9717,0.6376,0.0750,1.0000
4,0.5,0.3,nwkr_laplace,221,5,764,10,0.985,0.9779,0.9567,0.9672,0.6543,0.0874,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,1.0,0.3,nwkr_gaussian,221,5,764,10,0.985,0.9779,0.9567,0.9672,0.6376,0.0750,1.0000
84,1.0,0.3,nwkr_laplace,160,5,764,71,0.924,0.9697,0.6926,0.8081,0.6543,0.0874,1.0000
85,1.0,0.3,poly_deg1,27,92,677,204,0.704,0.2269,0.1169,0.1543,0.4802,0.1541,0.7500
86,1.0,0.3,poly_deg2,24,63,706,207,0.730,0.2759,0.1039,0.1509,0.5256,0.1328,0.7500


In [11]:
def performance_analysis(
    results_path: str,
    iou_thresh: float = 0.75,
    score_thresh: float = 0.3,
    out_dir: str = "data/score_experiment_full",
) -> pd.DataFrame:
    os.makedirs(out_dir, exist_ok=True)

    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} rows, {df['method'].nunique()} methods, "
          f"{df['label'].sum()} positive, {(~df['label']).sum()} negative")

    summary_rows = []
    for method, sub in df.groupby("method"):
        pos_rows = sub[sub["label"] == True]
        neg_rows = sub[sub["label"] == False]

        tp = int(((pos_rows["score"] >= score_thresh) &
                    (pos_rows["iou_val"] >= iou_thresh)).sum())
        fn = int(len(pos_rows) - tp)
        fp = int((neg_rows["score"] >= score_thresh).sum())
        tn = int(len(neg_rows) - fp)


        total     = tp + fp + tn + fn
        accuracy  = (tp + tn) / total        if total > 0   else 0.0
        precision = tp / (tp + fp)           if tp + fp > 0 else 0.0
        recall    = tp / (tp + fn)           if tp + fn > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                        if precision + recall > 0 else 0.0)

        summary_rows.append({
            "iou_thresh":     iou_thresh,
            "score_thresh":   score_thresh,
            "method":         method,
            "tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "accuracy":       round(accuracy,  4),
            "precision":      round(precision, 4),
            "recall":         round(recall,    4),
            "f1":             round(f1,        4),
            "mean_score_pos": round(float(pos_rows["score"].mean()), 4),
            "mean_score_neg": round(float(neg_rows["score"].mean()), 4),
            "median_iou_pos": round(float(pos_rows["iou_val"].median()), 4),
            "mean_time_ms":     round(float(sub["time_ms"].mean()),       3),
            "median_time_ms":   round(float(sub["time_ms"].median()),     3),
        })

    results = pd.DataFrame(summary_rows)

    for metric in ["f1", "precision", "recall", "accuracy"]:
        pivot = results.pivot(index="iou_thresh", columns="method", values=metric)
        print(f"\n=== {metric.upper()} ===")
        print(pivot.round(3).to_string())

    out_path = os.path.join(out_dir, f"performace_summary_{score_thresh:.2f}_{iou_thresh:.2f}.csv")
    results.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")
    return results


# --- Run ---
performance_analysis(
    results_path = "data/score_experiment_full/score_experiment_results.csv",
    iou_thresh   = 0.75,
    score_thresh = 0.3,
    out_dir      = "data/score_experiment_full",
)

Loaded 272167 rows, 7 methods, 1617 positive, 270550 negative

=== F1 ===
method       capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2
iou_thresh                                                                        
0.75        0.035  0.189  0.068          0.689         0.575      0.052      0.074

=== PRECISION ===
method       capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2
iou_thresh                                                                        
0.75        0.025  0.121  0.037          0.538         0.425      0.028       0.04

=== RECALL ===
method       capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2
iou_thresh                                                                        
0.75        0.061  0.429  0.429          0.961         0.887      0.489      0.528

=== ACCURACY ===
method      capa    lrt   mean  nwkr_gaussian  nwkr_laplace  poly_deg1  poly_deg2
iou_thresh                                  

,iou_thresh,score_thresh,method,tp,fp,tn,fn,accuracy,precision,recall,f1,mean_score_pos,mean_score_neg,median_iou_pos,mean_time_ms,median_time_ms
0,0.75,0.3,capa,14,554,38096,217,0.9802,0.0246,0.0606,0.0350,0.1506,0.0260,0.1695,8.637,5.419
1,0.75,0.3,lrt,99,718,37932,132,0.9781,0.1212,0.4286,0.1889,0.3452,0.0651,0.8000,756.310,182.626
2,0.75,0.3,mean,99,2561,36089,132,0.9307,0.0372,0.4286,0.0685,0.4036,0.1502,0.8333,394.341,94.944
3,0.75,0.3,nwkr_gaussian,222,191,38459,9,0.9949,0.5375,0.9610,0.6894,0.6376,0.0772,1.0000,190.254,45.235
4,0.75,0.3,nwkr_laplace,205,277,38373,26,0.9922,0.4253,0.8874,0.5750,0.6543,0.0912,1.0000,185.503,45.051
5,0.75,0.3,poly_deg1,113,3980,34670,118,0.8946,0.0276,0.4892,0.0523,0.4802,0.1547,0.7500,9200.581,2213.645
6,0.75,0.3,poly_deg2,122,2932,35718,109,0.9218,0.0399,0.5281,0.0743,0.5256,0.1343,0.7500,11181.243,2697.281
